# NI-DAQ alignment and Pynapple event analysis

Doric CAM pulses align clocks; Doric supplies primary licks; NI-DAQ supplies Ensure and other behavioral events.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pynapple as nap
from fluopulse_analysis import plot_aligned_session, process_aligned_session

In [ ]:
RUN = 4
DORIC_FILE = Path.home() / "Documents" / "SC81_260923" / f"260923_SC81_run{RUN}_0000.doric"
NIDAQ_FOLDER = Path.home() / "Documents" / "nidaq" / "SC81_260923"
NIDAQ_FILE = NIDAQ_FOLDER / f"SC81-260923-{RUN:03d}-nidaq.mat"
RUNNING_FILE = NIDAQ_FOLDER / f"SC81-260923-{RUN:03d}-running.mat"
for path in (DORIC_FILE, NIDAQ_FILE):
    if not path.is_file():
        raise FileNotFoundError(path)

In [ ]:
session = process_aligned_session(DORIC_FILE, NIDAQ_FILE, running_path=RUNNING_FILE)
print(f"Matched CAM pulses: {session.alignment.matched_pulses}")
print(f"Drift: {session.alignment.drift_ppm:.3f} ppm")
print(f"Residual RMS: {1000 * session.alignment.rms_residual_seconds:.3f} ms")
print(f"NI-DAQ coverage: {100 * session.nidaq_coverage_fraction:.2f}%")
print(f"Doric/NI lick counts: {session.doric_licks.onset_times.size}/{session.nidaq_licks.onset_times.size}")
print(f"Ensure count: {session.ensure_pulses.onset_times.size}")
figure = plot_aligned_session(session)
plt.show()

In [ ]:
data = session.to_pynapple()
EVENT_NAME = "licks"  # Change to "ensure" when solenoid events are present.
events = data[EVENT_NAME]
if len(events) == 0:
    raise ValueError(f"No {EVENT_NAME} events are present in this run")
tau_trials = nap.compute_perievent(data["tau"], events, window=(-2, 5), time_unit="s")
relative_time = np.asarray(tau_trials.t)
values = np.asarray(tau_trials)
if values.ndim == 1:
    values = values[:, None]
delta = values - np.nanmean(values[relative_time < 0], axis=0)
plt.plot(relative_time, delta, color="0.75", alpha=0.4)
plt.plot(relative_time, np.nanmean(delta, axis=1), color="tab:blue", linewidth=2)
plt.axvline(0, color="tab:orange")
plt.xlabel(f"Time from {EVENT_NAME} (s)")
plt.ylabel("Delta tau (ns)")
plt.show()

In [ ]:
lick_times = np.asarray(session.doric_licks.onset_times)

# A new bout begins following a gap greater than one second.
MAX_INTERLICK_GAP = 1.0

bout_starts = np.r_[0, np.flatnonzero(np.diff(lick_times) > MAX_INTERLICK_GAP) + 1]
bout_stops = np.r_[bout_starts[1:], len(lick_times)]

bout_onsets = lick_times[bout_starts]
bout_ends = lick_times[bout_stops - 1]
bout_counts = bout_stops - bout_starts

# Find the largest bout beginning near two minutes.
candidates = np.flatnonzero(
    (bout_onsets >= 90) & (bout_onsets <= 150)
)
major_bout = candidates[np.argmax(bout_counts[candidates])]

bout_onset = bout_onsets[major_bout]

print(f"Bout onset: {bout_onset:.3f} s")
print(f"Bout end: {bout_ends[major_bout]:.3f} s")
print(f"Licks: {bout_counts[major_bout]}")

In [ ]:
bout_event = nap.Ts(
    t=np.array([bout_onset]),
    time_units="s",
    time_support=data["tau"].time_support,
)

tau_around_bout = nap.compute_perievent(
    data["tau"],
    bout_event,
    window=(-20, 60),
    time_unit="s",
)

relative_time = np.asarray(tau_around_bout.t)
tau_values = np.asarray(tau_around_bout).squeeze()

# Subtract the mean lifetime during the 10 seconds before bout onset.
baseline = np.nanmean(
    tau_values[(relative_time >= -10) & (relative_time < 0)]
)
delta_tau = tau_values - baseline

relative_licks = lick_times - bout_onset
relative_licks = relative_licks[
    (relative_licks >= -20) & (relative_licks <= 60)
]

figure, axes = plt.subplots(
    2, 1, figsize=(11, 6), sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
)

axes[0].plot(relative_time, delta_tau)
axes[0].axvline(0, color="tab:orange", linestyle="--")
axes[0].set_ylabel("Δ tau (ns)")
axes[0].set_title("Lifetime aligned to major lick-bout onset")

axes[1].eventplot(relative_licks, colors="black")
axes[1].axvline(0, color="tab:orange", linestyle="--")
axes[1].set_ylabel("Licks")
axes[1].set_xlabel("Time from bout onset (s)")

plt.show()